# Setup de entorno uv para notebooks_clase

Este notebook crea un entorno aislado para ejecutar notebooks que vayas subiendo a `notebooks_clase`.

## Que hace
1. Detecta el root del repositorio.
2. Crea (o actualiza) un entorno virtual con `uv`.
3. Instala un kernel de Jupyter apuntando a ese entorno.
4. Deja un bloque para instalar paquetes extra cuando subas notebooks nuevos.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("No se encontro pyproject.toml al buscar el root del repo.")


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOKS_DIR = REPO_ROOT / "notebooks_clase"
VENV_DIR = NOTEBOOKS_DIR / ".venv"
KERNEL_NAME = "uv-notebooks-clase"
KERNEL_DISPLAY_NAME = "Python (uv notebooks_clase)"
PYTHON_VERSION = "3.12"
PYTHON_BIN = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

print(f"Repo root: {REPO_ROOT}")
print(f"Directorio notebooks: {NOTEBOOKS_DIR}")
print(f"Entorno uv: {VENV_DIR}")
print(f"Kernel: {KERNEL_NAME} -> {KERNEL_DISPLAY_NAME}")


In [ ]:
import shutil
import subprocess


def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if shutil.which("uv") is None:
    raise RuntimeError(
        "No se encontro uv en PATH. Instala uv antes de continuar: https://docs.astral.sh/uv/"
    )

NOTEBOOKS_DIR.mkdir(parents=True, exist_ok=True)
if VENV_DIR.exists():
    print(f"El entorno ya existe en {VENV_DIR}. Si quieres recrearlo, borra la carpeta y reejecuta.")
else:
    run(["uv", "venv", str(VENV_DIR), "--python", PYTHON_VERSION])


In [ ]:
run([
    "uv",
    "pip",
    "install",
    "--python",
    str(PYTHON_BIN),
    "ipykernel",
    "jupyterlab",
    "notebook",
])

run([
    str(PYTHON_BIN),
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    KERNEL_NAME,
    "--display-name",
    KERNEL_DISPLAY_NAME,
])

print("Listo. Ahora selecciona el kernel:", KERNEL_DISPLAY_NAME)


In [ ]:
# Agrega aqui dependencias nuevas cuando subas notebooks.
# Ejemplo: EXTRA_PACKAGES = ["pandas", "matplotlib", "scikit-learn"]
EXTRA_PACKAGES = []

if EXTRA_PACKAGES:
    run(["uv", "pip", "install", "--python", str(PYTHON_BIN), *EXTRA_PACKAGES])
    print("Paquetes instalados:", ", ".join(EXTRA_PACKAGES))
else:
    print("No hay paquetes extra por instalar.")


In [ ]:
run(["uv", "pip", "list", "--python", str(PYTHON_BIN)])


Si quieres recrear el entorno manualmente basta con que lances: 

```bash
uv venv /home/johndoe/Documents/software/dbx-ucm-productivizacion/notebooks_clase/.venv --python 3.11 --clear
```